In [ ]:
from __future__ import annotations
import json
import time
from pathlib import Path
import sys

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from transformers import AutoConfig, AutoModelForTokenClassification, AutoTokenizer

from pestclef.config import ExperimentConfig
from pestclef.data import load_documents
from pestclef.features import RelationSchema
from pestclef.mention_detection import MentionLexicon
from pestclef.modernbert import (
    ModernBertMentionDetector,
    ModernBertRelationModel,
    build_canonical_entities_from_mentions,
    build_relation_inference_examples,
    build_hybrid_span_candidates,
    resolve_torch_device,
)
from pestclef.submission import write_submission
from pestclef.schema import RelationEdge
from pestclef.data import deduplicate_edges

artifacts_dir = ROOT / 'artifacts' / 'modernbert_e2e_v9'
mention_model_dir = artifacts_dir / 'mention_model'
relation_model_dir = artifacts_dir / 'relation_model'
output_path = ROOT / 'submission_modernbert_e2e_v9.csv'

config = ExperimentConfig(
    project_root=ROOT,
    artifacts_dir=artifacts_dir,
    model_name='modernbert_staged',
    mention_cleanup_profile='strict_v1',
    mention_type_denylists_enabled=True,
    entity_alias_merge_strategy='heuristic_v1',
    relation_train_with_predicted_entities=True,
    relation_calibrate_on_predicted_entities=True,
    relation_pair_pruning_profile='precision_v1',
    device='auto',
)

mention_metadata = json.loads((mention_model_dir / 'metadata.json').read_text(encoding='utf-8'))
labels = list(mention_metadata['labels'])
label_to_id = {label: i for i, label in enumerate(labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
mention_tokenizer = AutoTokenizer.from_pretrained(mention_model_dir, use_fast=True)
mention_model_config = AutoConfig.from_pretrained(mention_model_dir)
mention_model = AutoModelForTokenClassification.from_pretrained(mention_model_dir, config=mention_model_config)
mention_thresholds = {k: float(v) for k, v in mention_metadata.get('mention_thresholds', {}).items()}
mention_detector = ModernBertMentionDetector(
    tokenizer=mention_tokenizer,
    model=mention_model,
    label_to_id=label_to_id,
    id_to_label=id_to_label,
    device=resolve_torch_device(config.device),
    config=config,
    mention_thresholds=mention_thresholds,
)
relation_model = ModernBertRelationModel.load(relation_model_dir, config)
train_documents = load_documents('train', config)
schema = RelationSchema.from_documents(train_documents)
mention_lexicon = MentionLexicon.from_documents(train_documents, config) if config.mention_hybrid_lexicon else None

test_documents = load_documents('test', config)
print(f'Loaded {len(test_documents)} test documents', flush=True)
start = time.time()
predictions = {}
for idx, document in enumerate(test_documents, start=1):
    _, _, blended_spans = build_hybrid_span_candidates(document, mention_detector, mention_lexicon)
    predicted_mentions = mention_detector.predict_mentions_from_spans(blended_spans, document)
    predicted_entities = build_canonical_entities_from_mentions(predicted_mentions, config=config)
    examples, entity_pairs = build_relation_inference_examples(document, predicted_entities, schema, config)
    edges = []
    for predicted_labels, (subject, obj) in zip(relation_model.predict_labels(examples), entity_pairs):
        for label in predicted_labels:
            if schema.is_valid_pair(label, subject.entity_type, obj.entity_type):
                edges.append(RelationEdge(subject=subject.canonical_form, predicate=label, object=obj.canonical_form))
    predictions[document.doc_id] = deduplicate_edges(edges)
    if idx % 10 == 0 or idx == len(test_documents):
        print(f'Processed {idx}/{len(test_documents)} docs in {time.time()-start:.1f}s', flush=True)

write_submission(predictions, [doc.doc_id for doc in test_documents], output_path)
rows = output_path.read_text(encoding='utf-8').splitlines()
print(json.dumps({'submission_path': str(output_path), 'documents': len(test_documents), 'csv_rows_including_header': len(rows), 'elapsed_sec': round(time.time()-start, 2)}, indent=2), flush=True)

/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 7594.89it/s]


Loaded 82 test documents
Processed 10/82 docs in 33.9s
Processed 20/82 docs in 65.0s
Processed 30/82 docs in 90.7s
Processed 40/82 docs in 136.5s
Processed 50/82 docs in 160.1s
Processed 60/82 docs in 191.6s
Processed 70/82 docs in 222.5s
Processed 80/82 docs in 246.7s
Processed 82/82 docs in 252.6s
{
  "submission_path": "/Users/mihai.radulescu/Documents/Master NLP/Sem2/BioNLP/PestCLEF2026/PestCLEF2026/submission_modernbert_e2e_v9.csv",
  "documents": 82,
  "csv_rows_including_header": 83,
  "elapsed_sec": 252.57
}


NameError: name 'PY' is not defined